In [4]:
import re
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [5]:
# 1. Inisialisasi tools Sastrawi (stopword remover & stemmer)
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()


In [6]:
# 2. Dataset: minimal 5 dokumen Bahasa Indonesia
#    Tema: teknologi, komputer, pendidikan
documents = [
    # Dokumen 1
    "Fakultas Teknik UNM mengumumkan bahwa pendaftaran mahasiswa baru "
    "untuk Program Studi Teknik Komputer telah dibuka mulai tanggal 1 "
    "Agustus 2026! Calon mahasiswa dapat mendaftar secara online melalui "
    "website resmi kampus.",

    # Dokumen 2
    "Perkembangan teknologi kecerdasan buatan (AI) semakin pesat dalam "
    "beberapa tahun terakhir. Banyak perusahaan teknologi besar mulai "
    "mengimplementasikan AI untuk meningkatkan efisiensi kerja mereka, "
    "seperti Google, Microsoft, dan Amazon.",

    # Dokumen 3
    "Sistem temu kembali informasi (Information Retrieval) merupakan "
    "salah satu cabang ilmu komputer yang mempelajari bagaimana cara "
    "menemukan dokumen yang relevan dari kumpulan data yang sangat besar "
    "dengan cepat dan akurat.",

    # Dokumen 4
    "Jurusan Teknik Komputer akan mengadakan seminar nasional bertema "
    "'Transformasi Digital di Era Society 5.0' pada tanggal 15 September "
    "2026 di Auditorium Fakultas Teknik. Seminar ini terbuka untuk umum "
    "dan gratis!",

    # Dokumen 5
    "Pembelajaran mesin atau machine learning adalah cabang dari "
    "kecerdasan buatan yang memungkinkan komputer belajar dari data "
    "tanpa harus diprogram secara eksplisit oleh manusia.",
]


In [7]:
# 3. Fungsi preprocess_text(text)
#    Urutan: case folding -> cleaning -> tokenisasi ->
#            stopwords removal -> stemming
def preprocess_text(text: str):
    """
    Melakukan pipeline preprocessing teks Bahasa Indonesia.
    Mengembalikan dict berisi hasil setiap tahap serta token akhir.
    """
    # a. Case folding -> ubah semua huruf menjadi huruf kecil
    text_casefold = text.lower()

    # b. Cleaning -> hapus angka, tanda baca, dan karakter khusus
    #    Hanya menyisakan huruf a-z dan spasi
    text_clean = re.sub(r"[^a-z\s]", " ", text_casefold)
    text_clean = re.sub(r"\s+", " ", text_clean).strip()

    # c. Tokenisasi -> pecah menjadi kata per kata
    tokens_before = text_clean.split()

    # d. Stopwords removal (Sastrawi bekerja pada string, bukan list token,
    #    sehingga kita gabungkan token menjadi kalimat terlebih dahulu)
    text_no_stopwords = stopword_remover.remove(" ".join(tokens_before))
    tokens_no_stopwords = text_no_stopwords.split()

    # e. Stemming -> ubah setiap kata ke bentuk dasarnya
    text_stemmed = stemmer.stem(" ".join(tokens_no_stopwords))
    tokens_final = text_stemmed.split()

    return {
        "original": text,
        "case_folded": text_casefold,
        "cleaned": text_clean,
        "tokens_before": tokens_before,
        "tokens_after_stopwords": tokens_no_stopwords,
        "tokens_final": tokens_final,
        "text_final": " ".join(tokens_final),
    }

In [8]:
# 4. Terapkan fungsi pada ke-5 dokumen
results = [preprocess_text(doc) for doc in documents]

In [9]:
# 5. Tampilkan perbandingan sebelum & sesudah (2 dokumen lengkap)
print("=" * 70)
print("PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING (Dokumen 1 & 2)")
print("=" * 70)

for i in [0, 1]:
    r = results[i]
    print(f"\n--- Dokumen {i + 1} ---")
    print(f"[SEBELUM] : {r['original']}")
    print(f"[SESUDAH] : {r['text_final']}")
    print(f"Jumlah token sebelum : {len(r['tokens_before'])}")
    print(f"Jumlah token sesudah : {len(r['tokens_final'])}")

# Ringkasan singkat untuk dokumen 3-5 juga (opsional, agar semua terlihat)
print("\n" + "=" * 70)
print("RINGKASAN SINGKAT DOKUMEN 3-5")
print("=" * 70)
for i in [2, 3, 4]:
    r = results[i]
    print(f"\nDokumen {i + 1}:")
    print(f"  Sebelum : {r['original'][:80]}...")
    print(f"  Sesudah : {r['text_final']}")


PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING (Dokumen 1 & 2)

--- Dokumen 1 ---
[SEBELUM] : Fakultas Teknik UNM mengumumkan bahwa pendaftaran mahasiswa baru untuk Program Studi Teknik Komputer telah dibuka mulai tanggal 1 Agustus 2026! Calon mahasiswa dapat mendaftar secara online melalui website resmi kampus.
[SESUDAH] : fakultas teknik unm umum daftar mahasiswa baru program studi teknik komputer buka mulai tanggal agustus calon mahasiswa daftar online lalu website resmi kampus
Jumlah token sebelum : 28
Jumlah token sesudah : 23

--- Dokumen 2 ---
[SEBELUM] : Perkembangan teknologi kecerdasan buatan (AI) semakin pesat dalam beberapa tahun terakhir. Banyak perusahaan teknologi besar mulai mengimplementasikan AI untuk meningkatkan efisiensi kerja mereka, seperti Google, Microsoft, dan Amazon.
[SESUDAH] : kembang teknologi cerdas buat ai makin pesat beberapa tahun akhir banyak usaha teknologi besar mulai implementasi ai tingkat efisiensi kerja seperti google microsoft amazon
Jumlah tok

In [10]:
# 6. Hitung statistik jumlah token & persentase pengurangan
print("\n" + "=" * 70)
print("STATISTIK JUMLAH TOKEN")
print("=" * 70)

import pandas as pd  # hanya untuk menampilkan tabel yang rapi (opsional)

stat_rows = []
total_before, total_after = 0, 0

for i, r in enumerate(results, start=1):
    n_before = len(r["tokens_before"])
    n_after = len(r["tokens_final"])
    reduction_pct = (n_before - n_after) / n_before * 100 if n_before else 0

    stat_rows.append({
        "Dokumen": f"Dok {i}",
        "Token Sebelum": n_before,
        "Token Sesudah": n_after,
        "Pengurangan (%)": round(reduction_pct, 2),
    })

    total_before += n_before
    total_after += n_after

stat_df = pd.DataFrame(stat_rows)
print(stat_df.to_string(index=False))

total_reduction_pct = (total_before - total_after) / total_before * 100
print(f"\nTotal token sebelum preprocessing : {total_before}")
print(f"Total token sesudah preprocessing : {total_after}")
print(f"Total persentase pengurangan token : {total_reduction_pct:.2f}%")


STATISTIK JUMLAH TOKEN
Dokumen  Token Sebelum  Token Sesudah  Pengurangan (%)
  Dok 1             28             23            17.86
  Dok 2             28             24            14.29
  Dok 3             30             23            23.33
  Dok 4             27             20            25.93
  Dok 5             23             15            34.78

Total token sebelum preprocessing : 136
Total token sesudah preprocessing : 105
Total persentase pengurangan token : 22.79%


ANALISIS SINGKAT:
Proses preprocessing (case folding, cleaning, tokenisasi, stopwords removal,
dan stemming) terbukti mampu mengurangi jumlah token secara signifikan tanpa
menghilangkan makna inti dari dokumen. Penghapusan stopwords menghilangkan
kata-kata yang sering muncul namun tidak memiliki nilai diskriminatif dalam
pencarian (seperti "yang", "untuk", "dan"), sementara stemming menyatukan
berbagai bentuk imbuhan kata menjadi satu kata dasar (misalnya "mendaftar"
dan "pendaftaran" menjadi bentuk dasar yang sama). Dampaknya bagi sistem
Information Retrieval (IR) adalah data menjadi lebih ringkas, konsisten, dan
lebih mudah dicocokkan (matching) antara query dan dokumen, sehingga dapat
meningkatkan efisiensi komputasi (index lebih kecil) dan relevansi hasil
pencarian, karena term yang secara semantik sama tidak lagi dianggap sebagai
term yang berbeda.
